# Plant Leaf Disease Detection — Hybrid DL + ML Experiment
**Paper:** Sujatha et al. (2025) — *Advancing plant leaf disease detection integrating machine learning and deep learning*  
**Journal:** Scientific Reports 15:11552 | DOI: 10.1038/s41598-024-72197-2  
**Framework:** PyTorch  
**Approach:** Frozen backbone (VGG19 / InceptionV3) → Feature Extraction → ML Classifier (SVM, kNN, RF, AdaBoost, DT)

| Scenario | Augmentation | Backbones | ML Classifiers |
|----------|-------------|-----------|----------------|
| Baseline | None | VGG19, InceptionV3 | SVM, kNN, RF, AdaBoost, DT |
| Augmented | Rotation + Zoom + HFlip + ColorJitter | VGG19, InceptionV3 | SVM, kNN, RF, AdaBoost, DT |

In [1]:
!pip install -q openpyxl Pillow torchinfo

In [2]:
import os, warnings, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torchinfo import summary

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)
from sklearn.model_selection import learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'PyTorch {torch.__version__} | Device: {DEVICE} | GPUs: {n_gpus}')

PyTorch 2.10.0+cu128 | Device: cuda


In [7]:
# ─── USER CONFIGURATION ───────────────────────────────────────────────────────
DATASET_NAME   = '/kaggle/input/datasets/tminhhi/finaldata/dataset_smartcrop_new'          # ← Thay tên Kaggle dataset của bạn
DATASET_PATH   = f'{DATASET_NAME}'
TRAIN_DIR      = os.path.join(DATASET_PATH, 'train')
VAL_DIR        = os.path.join(DATASET_PATH, 'val')
TEST_DIR       = os.path.join(DATASET_PATH, 'test')
PAPER_CITATION = 'Sujatha et al. (2025) - Scientific Reports 15:11552'
# ─────────────────────────────────────────────────────────────────────────────

BATCH_SIZE  = 32
NUM_WORKERS = 2

# ML Hyperparameters (không có trong paper → giá trị chuẩn)
SVM_PARAMS      = {'C': 1.0, 'kernel': 'rbf', 'probability': True, 'random_state': 42}
KNN_PARAMS      = {'n_neighbors': 5, 'metric': 'euclidean', 'n_jobs': -1}
RF_PARAMS       = {'n_estimators': 100, 'random_state': 42, 'n_jobs': -1}
ADABOOST_PARAMS = {'n_estimators': 100, 'random_state': 42}
DT_PARAMS       = {'random_state': 42}

IMG_VGG19     = (224, 224)
IMG_INCEPTION = (299, 299)
MEAN          = [0.485, 0.456, 0.406]
STD           = [0.229, 0.224, 0.225]

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded.')
print(f'  TRAIN: {TRAIN_DIR}')
print(f'  VAL  : {VAL_DIR}')
print(f'  TEST : {TEST_DIR}')

Config loaded.
  TRAIN: /kaggle/input/datasets/tminhhi/finaldata/dataset_smartcrop_new/train
  VAL  : /kaggle/input/datasets/tminhhi/finaldata/dataset_smartcrop_new/val
  TEST : /kaggle/input/datasets/tminhhi/finaldata/dataset_smartcrop_new/test


In [8]:
# ── Dataset Exploration ────────────────────────────────────────────────────────
def explore_dataset(train_dir, val_dir, test_dir):
    class_names = sorted(os.listdir(train_dir))
    splits = {'Train': train_dir, 'Val': val_dir, 'Test': test_dir}
    all_counts = {}
    for split, d in splits.items():
        counts = {}
        for cls in class_names:
            p = os.path.join(d, cls)
            if os.path.isdir(p):
                counts[cls] = len([f for f in os.listdir(p)
                                   if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
        all_counts[split] = counts
        total = sum(counts.values())
        print(f'{split:5s}: {total:5d} images | ' +
              ' | '.join(f'{c}:{n}' for c, n in counts.items()))

    colors = plt.cm.tab20(np.linspace(0, 1, len(class_names)))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (split, counts) in zip(axes, all_counts.items()):
        bars = ax.bar(list(counts.keys()), list(counts.values()), color=colors)
        ax.set_title(f'{split} Set Distribution', fontsize=13, fontweight='bold')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=45)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    str(int(bar.get_height())), ha='center', va='bottom', fontsize=8)
    plt.suptitle('Dataset Statistics', fontsize=15, fontweight='bold')
    plt.tight_layout()
    path = f'{OUTPUT_DIR}/dataset_statistics.png'
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {path}')
    return class_names

CLASS_NAMES = explore_dataset(TRAIN_DIR, VAL_DIR, TEST_DIR)
NUM_CLASSES = len(CLASS_NAMES)
print(f'\nClasses ({NUM_CLASSES}): {CLASS_NAMES}')

Train:  1597 images | anthracnose:264 | healthy:334 | leaf_curl:318 | leaf_spot:326 | nutrient_deficiency:355
Val  :   491 images | anthracnose:81 | healthy:102 | leaf_curl:98 | leaf_spot:101 | nutrient_deficiency:109
Test :   373 images | anthracnose:62 | healthy:78 | leaf_curl:74 | leaf_spot:76 | nutrient_deficiency:83
Saved: /kaggle/working/dataset_statistics.png

Classes (5): ['anthracnose', 'healthy', 'leaf_curl', 'leaf_spot', 'nutrient_deficiency']


In [9]:
# ── Transforms ────────────────────────────────────────────────────────────────
# Scenario 1: No Augmentation
T_NOAUG_VGG = transforms.Compose([
    transforms.Resize(IMG_VGG19),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
T_NOAUG_INC = transforms.Compose([
    transforms.Resize(IMG_INCEPTION),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Scenario 2: 4 Augmentation Techniques
T_AUG_VGG = transforms.Compose([
    transforms.Resize(IMG_VGG19),
    transforms.RandomRotation(degrees=20),                           # 1. Rotation
    transforms.RandomResizedCrop(IMG_VGG19, scale=(0.8, 1.0)),      # 2. Zoom/Scale
    transforms.RandomHorizontalFlip(p=0.5),                          # 3. Horizontal Flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2,             # 4. ColorJitter
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
T_AUG_INC = transforms.Compose([
    transforms.Resize(IMG_INCEPTION),
    transforms.RandomRotation(degrees=20),
    transforms.RandomResizedCrop(IMG_INCEPTION, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
print('Transforms defined.')

Transforms defined.


In [12]:
# ── Feature Extractors (Frozen Backbones) ─────────────────────────────────────
class VGG19Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.avgpool  = base.avgpool
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return x.flatten(1)   # (B, 25088)


class InceptionV3Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.inception_v3(
            weights=models.Inception_V3_Weights.IMAGENET1K_V1,
            aux_logits=True
        )
        base.fc = nn.Identity()
        self.model = base
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.model(x)   # (B, 2048)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

VGG19_PARAMS     = count_params(VGG19Extractor())
INCEPTION_PARAMS = count_params(InceptionV3Extractor())
print(f'VGG19       total params: {VGG19_PARAMS:,}')
print(f'InceptionV3 total params: {INCEPTION_PARAMS:,}')

# Model summaries
print('
=== VGG19 Extractor Summary ===')
_vgg_tmp = VGG19Extractor()
summary(_vgg_tmp, input_size=(1, 3, 224, 224), device='cpu')
del _vgg_tmp

print('
=== InceptionV3 Extractor Summary ===')
_inc_tmp = InceptionV3Extractor()
summary(_inc_tmp, input_size=(1, 3, 299, 299), device='cpu')
del _inc_tmp

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 182MB/s] 


VGG19       total params: 20,024,384
InceptionV3 total params: 25,112,264


In [13]:
# ── Utility Functions ──────────────────────────────────────────────────────────
def extract_features(extractor, dataloader):
    extractor.eval().to(DEVICE)
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in dataloader:
            out = extractor(imgs.to(DEVICE))
            feats.append(out.cpu().numpy())
            labels.extend(lbls.numpy())
    return np.vstack(feats), np.array(labels)


def preprocess_features(X_tr, X_te):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)
    max_components = min(X_tr.shape[0], X_tr.shape[1])
    pca = PCA(n_components=min(0.95, float(max_components)), random_state=42)
    X_tr = pca.fit_transform(X_tr)
    X_te = pca.transform(X_te)
    var  = pca.explained_variance_ratio_.sum()
    print(f'    PCA: {X_tr.shape[1]}d | variance retained: {var:.2%}')
    return X_tr, X_te


def plot_learning_curve(clf, X_tr, y_tr, title, path):
    sizes, tr_sc, cv_sc = learning_curve(
        clf, X_tr, y_tr, cv=5, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(sizes, tr_sc.mean(1), 'o-', color='royalblue', label='Train')
    ax.fill_between(sizes, tr_sc.mean(1) - tr_sc.std(1),
                    tr_sc.mean(1) + tr_sc.std(1), alpha=0.1, color='royalblue')
    ax.plot(sizes, cv_sc.mean(1), 's-', color='darkorange', label='CV Val')
    ax.fill_between(sizes, cv_sc.mean(1) - cv_sc.std(1),
                    cv_sc.mean(1) + cv_sc.std(1), alpha=0.1, color='darkorange')
    ax.set_xlabel('Training Samples', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_confusion_matrix(y_true, y_pred, labels, title, path):
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    sz   = max(8, len(labels))
    fig, ax = plt.subplots(figsize=(sz, sz - 1))
    disp.plot(cmap='Blues', ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()


def run_scenario(backbone_name, total_params,
                 X_tr, y_tr, X_te, y_te,
                 scenario_tag, class_names, out_dir):
    print(f'  Preprocessing features for {backbone_name}...')
    X_tr_p, X_te_p = preprocess_features(X_tr, X_te)
    clf_map = {
        'SVM'      : SVC(**SVM_PARAMS),
        'kNN'      : KNeighborsClassifier(**KNN_PARAMS),
        'RF'       : RandomForestClassifier(**RF_PARAMS),
        'AdaBoost' : AdaBoostClassifier(**ADABOOST_PARAMS),
        'DT'       : DecisionTreeClassifier(**DT_PARAMS),
    }
    results = []
    for clf_name, clf in clf_map.items():
        print(f'  [{backbone_name}+{clf_name}]', end=' ', flush=True)
        clf.fit(X_tr_p, y_tr)
        y_pred = clf.predict(X_te_p)
        acc = accuracy_score(y_te, y_pred)
        f1  = f1_score(y_te, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_te, y_pred, average='macro', zero_division=0)

        tag  = f'{backbone_name}_{clf_name}_{scenario_tag}'
        lc_p = f'{out_dir}/{tag}_LC.png'
        cm_p = f'{out_dir}/{tag}_CM.png'
        plot_learning_curve(
            clf, X_tr_p, y_tr,
            f'Learning Curve\n{backbone_name} + {clf_name} ({scenario_tag})', lc_p
        )
        plot_confusion_matrix(
            y_te, y_pred, class_names,
            f'Confusion Matrix\n{backbone_name} + {clf_name} ({scenario_tag})', cm_p
        )
        results.append({
            'model_name'   : f'{backbone_name} + {clf_name}',
            'accuracy'     : acc,
            'f1_macro'     : f1,
            'recall_macro' : rec,
            'total_params' : total_params,
            'lc_path'      : lc_p,
            'cm_path'      : cm_p,
        })
        print(f'Acc={acc:.4f} | F1={f1:.4f} | Recall={rec:.4f}')
        print(classification_report(y_te, y_pred, target_names=class_names, zero_division=0))
    return results


print('Utility functions ready.')

Utility functions ready.


---
## SCENARIO 1: NO AUGMENTATION (Baseline)
Transform: `Resize → ToTensor → Normalize`

In [14]:
# Load DataLoaders
dl_tr_vgg = DataLoader(ImageFolder(TRAIN_DIR, T_NOAUG_VGG), BATCH_SIZE,
                       shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
dl_te_vgg = DataLoader(ImageFolder(TEST_DIR,  T_NOAUG_VGG), BATCH_SIZE,
                       shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
dl_tr_inc = DataLoader(ImageFolder(TRAIN_DIR, T_NOAUG_INC), BATCH_SIZE,
                       shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
dl_te_inc = DataLoader(ImageFolder(TEST_DIR,  T_NOAUG_INC), BATCH_SIZE,
                       shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Extract features
print('Extracting VGG19 features (No-Aug)...')
vgg_ext = VGG19Extractor()
X_tr_vgg_na, y_tr_vgg_na = extract_features(vgg_ext, dl_tr_vgg)
X_te_vgg_na, y_te_vgg_na = extract_features(vgg_ext, dl_te_vgg)
del vgg_ext; torch.cuda.empty_cache()
print(f'  train={X_tr_vgg_na.shape}, test={X_te_vgg_na.shape}')

print('Extracting InceptionV3 features (No-Aug)...')
inc_ext = InceptionV3Extractor()
X_tr_inc_na, y_tr_inc_na = extract_features(inc_ext, dl_tr_inc)
X_te_inc_na, y_te_inc_na = extract_features(inc_ext, dl_te_inc)
del inc_ext; torch.cuda.empty_cache()
print(f'  train={X_tr_inc_na.shape}, test={X_te_inc_na.shape}')

Extracting VGG19 features (No-Aug)...
  train=(1597, 25088), test=(373, 25088)
Extracting InceptionV3 features (No-Aug)...
  train=(1597, 2048), test=(373, 2048)


In [15]:
print('=' * 65)
print('TRAINING SCENARIO 1: NO AUGMENTATION')
print('=' * 65)
BASELINE_RESULTS = []
BASELINE_RESULTS += run_scenario('VGG19',       VGG19_PARAMS,
                                 X_tr_vgg_na, y_tr_vgg_na,
                                 X_te_vgg_na, y_te_vgg_na,
                                 'NoAug', CLASS_NAMES, OUTPUT_DIR)
BASELINE_RESULTS += run_scenario('InceptionV3', INCEPTION_PARAMS,
                                 X_tr_inc_na, y_tr_inc_na,
                                 X_te_inc_na, y_te_inc_na,
                                 'NoAug', CLASS_NAMES, OUTPUT_DIR)

# ── Summary Table ──────────────────────────────────────────────────────────────
BASELINE_LEARNING_CURVE_PATH  = BASELINE_RESULTS[0]['lc_path']  # best: VGG19+SVM
BASELINE_CONFUSION_MATRIX_PATH = BASELINE_RESULTS[0]['cm_path']
baseline_metrics = BASELINE_RESULTS[0]   # used by Excel export

print()
print(f'{"Model":<32} {"Accuracy":>10} {"F1 Macro":>10} {"Recall":>10}')
print('-' * 65)
for r in BASELINE_RESULTS:
    print(f'{r["model_name"]:<32} {r["accuracy"]:>10.4f} {r["f1_macro"]:>10.4f} {r["recall_macro"]:>10.4f}')

TRAINING SCENARIO 1: NO AUGMENTATION
  Preprocessing features for VGG19...
    PCA 256d | variance retained: 63.55%
  [VGG19+SVM] Acc=0.8445 | F1=0.8470 | Recall=0.8514
  [VGG19+kNN] Acc=0.6086 | F1=0.6174 | Recall=0.6161
  [VGG19+RF] Acc=0.8123 | F1=0.8175 | Recall=0.8205
  [VGG19+AdaBoost] Acc=0.6810 | F1=0.6941 | Recall=0.6928
  [VGG19+DT] Acc=0.6595 | F1=0.6695 | Recall=0.6699
  Preprocessing features for InceptionV3...
    PCA 256d | variance retained: 89.17%
  [InceptionV3+SVM] Acc=0.9142 | F1=0.9171 | Recall=0.9192
  [InceptionV3+kNN] Acc=0.8847 | F1=0.8887 | Recall=0.8900
  [InceptionV3+RF] Acc=0.8660 | F1=0.8694 | Recall=0.8722
  [InceptionV3+AdaBoost] Acc=0.7212 | F1=0.7321 | Recall=0.7341
  [InceptionV3+DT] Acc=0.7587 | F1=0.7662 | Recall=0.7673

Model                              Accuracy   F1 Macro     Recall
-----------------------------------------------------------------
VGG19 + SVM                          0.8445     0.8470     0.8514
VGG19 + kNN                       

---
## SCENARIO 2: AUGMENTATION
Transform (train only): `Resize → Rotation(20°) → Zoom/Scale(0.8-1.0) → HorizontalFlip → ColorJitter → ToTensor → Normalize`  
Transform (test): No-Aug (same as Scenario 1)

In [16]:
# Load DataLoaders — Aug on train, NoAug on test
dl_tr_vgg_a = DataLoader(ImageFolder(TRAIN_DIR, T_AUG_VGG), BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
dl_tr_inc_a = DataLoader(ImageFolder(TRAIN_DIR, T_AUG_INC), BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Extract features
print('Extracting VGG19 features (Aug)...')
vgg_ext = VGG19Extractor()
X_tr_vgg_a, y_tr_vgg_a = extract_features(vgg_ext, dl_tr_vgg_a)
del vgg_ext; torch.cuda.empty_cache()
print(f'  train={X_tr_vgg_a.shape}')

print('Extracting InceptionV3 features (Aug)...')
inc_ext = InceptionV3Extractor()
X_tr_inc_a, y_tr_inc_a = extract_features(inc_ext, dl_tr_inc_a)
del inc_ext; torch.cuda.empty_cache()
print(f'  train={X_tr_inc_a.shape}')

# Test features reuse No-Aug extractions from Scenario 1
print('Test features reused from Scenario 1 (No-Aug).')

Extracting VGG19 features (Aug)...
  train=(1597, 25088)
Extracting InceptionV3 features (Aug)...
  train=(1597, 2048)
Test features reused from Scenario 1 (No-Aug).


In [17]:
print('=' * 65)
print('TRAINING SCENARIO 2: WITH AUGMENTATION')
print('=' * 65)
AUG_RESULTS = []
AUG_RESULTS += run_scenario('VGG19',       VGG19_PARAMS,
                            X_tr_vgg_a, y_tr_vgg_a,
                            X_te_vgg_na, y_te_vgg_na,
                            'Aug', CLASS_NAMES, OUTPUT_DIR)
AUG_RESULTS += run_scenario('InceptionV3', INCEPTION_PARAMS,
                            X_tr_inc_a, y_tr_inc_a,
                            X_te_inc_na, y_te_inc_na,
                            'Aug', CLASS_NAMES, OUTPUT_DIR)

AUG_LEARNING_CURVE_PATH   = AUG_RESULTS[0]['lc_path']
AUG_CONFUSION_MATRIX_PATH = AUG_RESULTS[0]['cm_path']
aug_metrics = AUG_RESULTS[0]

print()
print(f'{"Model":<32} {"Accuracy":>10} {"F1 Macro":>10} {"Recall":>10}')
print('-' * 65)
for r in AUG_RESULTS:
    print(f'{r["model_name"]:<32} {r["accuracy"]:>10.4f} {r["f1_macro"]:>10.4f} {r["recall_macro"]:>10.4f}')

TRAINING SCENARIO 2: WITH AUGMENTATION
  Preprocessing features for VGG19...
    PCA 256d | variance retained: 54.94%
  [VGG19+SVM] Acc=0.8043 | F1=0.8111 | Recall=0.8153
  [VGG19+kNN] Acc=0.3378 | F1=0.3290 | Recall=0.3408
  [VGG19+RF] Acc=0.6756 | F1=0.6799 | Recall=0.6891
  [VGG19+AdaBoost] Acc=0.6408 | F1=0.6555 | Recall=0.6566
  [VGG19+DT] Acc=0.5094 | F1=0.5194 | Recall=0.5261
  Preprocessing features for InceptionV3...
    PCA 256d | variance retained: 87.89%
  [InceptionV3+SVM] Acc=0.8847 | F1=0.8882 | Recall=0.8911
  [InceptionV3+kNN] Acc=0.7802 | F1=0.7837 | Recall=0.7887
  [InceptionV3+RF] Acc=0.7989 | F1=0.8048 | Recall=0.8083
  [InceptionV3+AdaBoost] Acc=0.6300 | F1=0.6317 | Recall=0.6422
  [InceptionV3+DT] Acc=0.6542 | F1=0.6684 | Recall=0.6680

Model                              Accuracy   F1 Macro     Recall
-----------------------------------------------------------------
VGG19 + SVM                          0.8043     0.8111     0.8153
VGG19 + kNN                     

---
## EXCEL EXPORT
Xuất `results_summary.xlsx` với 2 sheet (Baseline + Augmented).  
Mỗi sheet có 10 hàng (VGG19×5 + InceptionV3×5), ảnh LC và CM nhúng trực tiếp.

In [19]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_PATH = f'{OUTPUT_DIR}/results_summary.xlsx'
HEADERS    = ['STT', 'Ten Model', 'Accuracy', 'F1-score (Macro)',
              'Recall (Macro)', 'Learning Curve', 'Confusion Matrix', 'Parameters']
COL_WIDTHS = [6, 38, 14, 18, 16, 42, 42, 22]
IMG_W, IMG_H = 310, 200
ROW_H        = 155  # points


def style_header(ws):
    fill = PatternFill('solid', fgColor='1F4E79')
    font = Font(bold=True, color='FFFFFF', size=11)
    for col, h in enumerate(HEADERS, 1):
        c = ws.cell(1, col, h)
        c.fill, c.font = fill, font
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[1].height = 28


def fill_sheet(ws, results):
    for i, cw in enumerate(COL_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(i)].width = cw
    style_header(ws)
    for idx, r in enumerate(results, 1):
        row = idx + 1
        ws.row_dimensions[row].height = ROW_H
        vals = [
            idx,
            f"{r['model_name']} ({PAPER_CITATION})",
            round(r['accuracy'], 4),
            round(r['f1_macro'], 4),
            round(r['recall_macro'], 4),
            '',   # LC image
            '',   # CM image
            f"{r['total_params']:,}",
        ]
        for col, v in enumerate(vals, 1):
            c = ws.cell(row, col, v)
            c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        if os.path.exists(r['lc_path']):
            img = XLImage(r['lc_path'])
            img.width, img.height = IMG_W, IMG_H
            ws.add_image(img, f'F{row}')
        if os.path.exists(r['cm_path']):
            img = XLImage(r['cm_path'])
            img.width, img.height = IMG_W, IMG_H
            ws.add_image(img, f'G{row}')


wb = Workbook()
wb.remove(wb.active)
ws1 = wb.create_sheet('Baseline')
ws2 = wb.create_sheet('Augmented')
fill_sheet(ws1, BASELINE_RESULTS)
fill_sheet(ws2, AUG_RESULTS)
wb.save(EXCEL_PATH)

print(f'Saved: {EXCEL_PATH}')
print(f'  Sheet Baseline  : {len(BASELINE_RESULTS)} rows')
print(f'  Sheet Augmented : {len(AUG_RESULTS)} rows')
print()
print('=== FINAL RESULTS SUMMARY ===')
print(f'{"Model":<32} {"Scenario":<12} {"Accuracy":>10} {"F1":>8} {"Recall":>8}')
print('-' * 75)
for r in BASELINE_RESULTS:
    print(f'{r["model_name"]:<32} {"NoAug":<12} {r["accuracy"]:>10.4f} {r["f1_macro"]:>8.4f} {r["recall_macro"]:>8.4f}')
for r in AUG_RESULTS:
    print(f'{r["model_name"]:<32} {"Aug":<12} {r["accuracy"]:>10.4f} {r["f1_macro"]:>8.4f} {r["recall_macro"]:>8.4f}')

Saved: /kaggle/working/results_summary.xlsx
  Sheet Baseline  : 10 rows
  Sheet Augmented : 10 rows

=== FINAL RESULTS SUMMARY ===
Model                            Scenario       Accuracy       F1   Recall
---------------------------------------------------------------------------
VGG19 + SVM                      NoAug            0.8445   0.8470   0.8514
VGG19 + kNN                      NoAug            0.6086   0.6174   0.6161
VGG19 + RF                       NoAug            0.8123   0.8175   0.8205
VGG19 + AdaBoost                 NoAug            0.6810   0.6941   0.6928
VGG19 + DT                       NoAug            0.6595   0.6695   0.6699
InceptionV3 + SVM                NoAug            0.9142   0.9171   0.9192
InceptionV3 + kNN                NoAug            0.8847   0.8887   0.8900
InceptionV3 + RF                 NoAug            0.8660   0.8694   0.8722
InceptionV3 + AdaBoost           NoAug            0.7212   0.7321   0.7341
InceptionV3 + DT                 NoAug     